### Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


In [2]:
df = pd.read_csv("medicaid-provider-spending.csv")
print(df.columns.tolist())  # column names

/var/folders/dj/z48f7b2j74j0qb_5_slvd9w80000gn/T/ipykernel_36322/2520810276.py:1: DtypeWarning: Columns (0: BILLING_PROVIDER_NPI_NUM) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("medicaid-provider-spending.csv")


['BILLING_PROVIDER_NPI_NUM', 'SERVICING_PROVIDER_NPI_NUM', 'HCPCS_CODE', 'CLAIM_FROM_MONTH', 'TOTAL_UNIQUE_BENEFICIARIES', 'TOTAL_CLAIMS', 'TOTAL_PAID']


### filter on codes provided by Clinicans

In [3]:
codes_df = pd.read_excel("Codes Manual.xlsx")

codes = (
    codes_df["CODE"]
    .astype(str)
    .str.strip()
    .dropna()
    .unique()
)

df["HCPCS_CODE"] = df["HCPCS_CODE"].astype(str).str.strip()

filtered_df = df[df["HCPCS_CODE"].isin(codes)].copy()

In [4]:
filtered_df["CLAIM_FROM_MONTH"] = pd.to_datetime(filtered_df["CLAIM_FROM_MONTH"])

filtered_df.sample(20)

,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID
155419786,1205474715,1790042455,96110,2021-05-01,17,17,178.20
148567356,1548395932,1265538219,93000,2022-03-01,15,15,230.49
22974996,1174503999,1003904020,88342,2022-02-01,52,62,5403.30
164887963,1811929151,1063679827,85025,2021-05-01,20,25,117.65
142309017,1699441907,1073995544,93010,2023-03-01,83,124,285.21
173342803,1952613416,1902888332,85025,2019-07-01,19,23,72.74
154766564,1942532791,1942532791,93000,2021-03-01,13,13,182.59
105859419,1821093402,1720370158,92552,2018-01-01,24,24,729.26
191881699,1285623009,NaN,93010,2020-05-01,19,20,0.00
185111052,1699769901,1457300287,85025,2023-10-01,13,13,20.97


### Check join status

In [5]:
codes_set = set(codes_df["CODE"].astype(str).str.strip())
claims_set = set(filtered_df["HCPCS_CODE"].astype(str).str.strip())

missing_codes = codes_set - claims_set

print("total codes: {}".format(len(codes_set)))
print("missing codes: {}".format(len(missing_codes)))


total codes: 288
missing codes: 63


In [6]:
missing_codes_df = (
    pd.DataFrame({"HCPCS_CODE": list(missing_codes)})
    .assign(HCPCS_CODE=lambda d: d["HCPCS_CODE"].astype(str).str.strip())
    .merge(
        codes_df.assign(CODE=codes_df["CODE"].astype(str).str.strip()),
        left_on="HCPCS_CODE",
        right_on="CODE",
        how="left"
    )
    .drop(columns=["CODE"])
)

missing_codes_df

,HCPCS_CODE,Description,Category,Unnamed: 3,Unnamed: 4
0,95726,Long-term EEG review and analysis (profession...,ADDITIONAL/ADJUNCT AUTONOMIC TESTING,NaN,NaN
1,27235,Percutaneous skeletal fixation of a femoral f...,NEGATIVE CONTROL,NaN,NaN
2,71252,CT Chest,CARDIAC IMAGING,NaN,NaN
3,38520,"Biopsy or excision of lymph node(s); open, de...",MISC,NaN,NaN
4,85257,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
...,...,...,...,...,...
58,85265,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
59,85259,Clotting factors (selective use),FIBRINOLYSIS/CLOT STABILITY,NaN,NaN
60,20225,Bone biopsy deep,MISC,NaN,NaN
61,75558,cardiovascular magnetic resonance imaging,CARDIAC IMAGING,NaN,NaN


### Join

In [7]:
filtered_df["HCPCS_CODE"] = filtered_df["HCPCS_CODE"].astype(str).str.strip()
codes_df["CODE"] = codes_df["CODE"].astype(str).str.strip()

retsef_meta = codes_df[["CODE", "Description"]].drop_duplicates()

filtered_with_meta = filtered_df.merge(
    retsef_meta,
    left_on="HCPCS_CODE",
    right_on="CODE",
    how="left"
).drop(columns=["CODE"])

monthly = (
    filtered_with_meta
    .groupby(["HCPCS_CODE", "CLAIM_FROM_MONTH"], as_index=False)
    .agg({
        "Description": "first",
        "TOTAL_PAID": "sum",
        "TOTAL_CLAIMS": "sum",
        "TOTAL_UNIQUE_BENEFICIARIES": "sum"
    })
    .sort_values(["HCPCS_CODE", "CLAIM_FROM_MONTH"])
)

monthly

,HCPCS_CODE,CLAIM_FROM_MONTH,Description,TOTAL_PAID,TOTAL_CLAIMS,TOTAL_UNIQUE_BENEFICIARIES
0,0295T,2018-03-01,Extended cardiac monitoring,50.58,14,14
1,0295T,2018-04-01,Extended cardiac monitoring,388.26,15,14
2,0295T,2018-05-01,Extended cardiac monitoring,0.00,14,14
3,0295T,2018-06-01,Extended cardiac monitoring,610.95,36,26
4,0295T,2018-08-01,Extended cardiac monitoring,0.00,33,32
...,...,...,...,...,...,...
14915,96146,2024-08-01,Psychological or neuropsychological test admi...,3072.62,537,451
14916,96146,2024-09-01,Psychological or neuropsychological test admi...,3020.83,536,450
14917,96146,2024-10-01,Psychological or neuropsychological test admi...,1367.78,465,386
14918,96146,2024-11-01,Psychological or neuropsychological test admi...,1388.73,304,241


### Pivot

In [8]:
claims_pivot = (
    monthly
    .pivot(index="CLAIM_FROM_MONTH", columns="HCPCS_CODE", values="TOTAL_CLAIMS")
    .sort_index()
)

claims_pivot

HCPCS_CODE,0295T,0296T,0297T,0298T,0464T,11100,11101,11102,11103,11104,...,96127,96130,96131,96132,96133,96136,96137,96138,96139,96146
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,45.0,566.0,88.0,NaN,10190.0,1047.0,NaN,NaN,NaN,...,146929.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-02-01,NaN,108.0,705.0,192.0,NaN,9802.0,1176.0,NaN,NaN,NaN,...,139342.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-03-01,14.0,112.0,908.0,276.0,NaN,11276.0,1254.0,NaN,NaN,NaN,...,160875.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-01,15.0,106.0,889.0,191.0,NaN,11231.0,1243.0,NaN,NaN,NaN,...,162667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-05-01,14.0,188.0,1048.0,183.0,NaN,12689.0,1431.0,NaN,NaN,NaN,...,170539.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11542.0,1197.0,1034.0,...,662917.0,20345.0,12987.0,8554.0,3660.0,12112.0,9383.0,8638.0,3581.0,537.0
2024-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9255.0,831.0,853.0,...,524272.0,21528.0,11842.0,8939.0,4105.0,11446.0,9098.0,7753.0,3315.0,536.0
2024-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9979.0,967.0,1070.0,...,550548.0,23736.0,11977.0,9617.0,4511.0,12180.0,9920.0,8350.0,3524.0,465.0


### Truncate to end of 2023

Claims for the most recent ~12 months are still being adjudicated, so the
series is cut off at December 2023.

In [9]:
full_months = pd.date_range(
    monthly["CLAIM_FROM_MONTH"].min(),
    monthly["CLAIM_FROM_MONTH"].max(),
    freq="MS"
)
claims_pivot = claims_pivot.reindex(full_months)
claims_pivot.index.name = "CLAIM_FROM_MONTH"
claims_pivot

HCPCS_CODE,0295T,0296T,0297T,0298T,0464T,11100,11101,11102,11103,11104,...,96127,96130,96131,96132,96133,96136,96137,96138,96139,96146
CLAIM_FROM_MONTH,,,,,,,,,,,,,,,,,,,,,
2018-01-01,NaN,45.0,566.0,88.0,NaN,10190.0,1047.0,NaN,NaN,NaN,...,146929.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-02-01,NaN,108.0,705.0,192.0,NaN,9802.0,1176.0,NaN,NaN,NaN,...,139342.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-03-01,14.0,112.0,908.0,276.0,NaN,11276.0,1254.0,NaN,NaN,NaN,...,160875.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-04-01,15.0,106.0,889.0,191.0,NaN,11231.0,1243.0,NaN,NaN,NaN,...,162667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-05-01,14.0,188.0,1048.0,183.0,NaN,12689.0,1431.0,NaN,NaN,NaN,...,170539.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11542.0,1197.0,1034.0,...,662917.0,20345.0,12987.0,8554.0,3660.0,12112.0,9383.0,8638.0,3581.0,537.0
2024-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9255.0,831.0,853.0,...,524272.0,21528.0,11842.0,8939.0,4105.0,11446.0,9098.0,7753.0,3315.0,536.0
2024-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9979.0,967.0,1070.0,...,550548.0,23736.0,11977.0,9617.0,4511.0,12180.0,9920.0,8350.0,3524.0,465.0


In [ ]:
cutoff = pd.Timestamp("2023-12-01")

claims_pivot = claims_pivot.loc[claims_pivot.index <= cutoff]

claims_pivot

In [11]:
claims_pivot.to_csv("claims_pivot.csv")

In [12]:
missing_codes_df.to_csv("missing_codes.csv", index=False)